## Dataset

For the learning purpose, I will be using Harry Potter books which are parsed into text format and this is available in the Kaggle or HuggingFace to freely use.

Kaggle Link: https://www.kaggle.com/datasets/rupanshukapoor/harry-potter-books

HuggingFace: https://huggingface.co/datasets/elricwan/HarryPotter

**I will personally be using HuggingFace as it is more convenient rather than downloading the dataset and storing in project directory**

In [ ]:
from datasets import load_dataset

ds = load_dataset("elricwan/HarryPotter")

In [126]:
# Explore the filenames in the dataset
for i in range(len(ds['train'])):
    print(f"Index {i+1}: {ds['train'][i]['filename']}")

Index 1: 1-Harry-Potter-and-the-Sorcerer’s-Stone.txt
Index 2: 2-Harry-Potter-and-the-Chamber-of-Secrets.txt
Index 3: 3-Harry-Potter-and-the-Prisoner-of-Azkaban.txt
Index 4: 4-Harry-Potter-and-the-Goblet-of-Fire.txt
Index 5: 5-Harry-Potter-and-the-Order-of-the-Phoenix.txt
Index 6: 6-Harry-Potter-and-the-Half-Blood-Prince.txt
Index 7: 7-Harry-Potter-and-the-Deathly-Hallows.txt
Index 8: Harry-Potter.txt


**When I skimmed through the dataset in HuggingFace, I learned that the 8th file, that is, `Harry-Potter.txt`, is the union of contents from all the other books**

This saves a step for us to iterate and combine all the conrents of the books.

In [128]:
# Find and extract the Harry-Potter.txt file (the 8th file with all content)
harry_potter_full = None

for i in range(len(ds['train'])):
    if ds['train'][i]['filename'] == 'Harry-Potter.txt':
        harry_potter_full = ds['train'][i]['content']
        print(f"Found 'Harry-Potter.txt' at index {i}")
        print(f"Content length: {len(harry_potter_full)} characters")
        print(f"\nFirst 200 characters:\n{harry_potter_full[:50]}")
        break

#  Use this as your raw_text
if harry_potter_full:
    raw_text = harry_potter_full
    print(f"\n✓ Successfully loaded Harry-Potter.txt into raw_text variable")
else:
    print("Harry-Potter.txt not found in dataset")

Found 'Harry-Potter.txt' at index 7
Content length: 6491209 characters

First 200 characters:
FOR JESSICA, WHO LOVES STORIES,

FOR ANNE, WHO LOV

✓ Successfully loaded Harry-Potter.txt into raw_text variable


### Step 1: Creating Tokens

<div class="alert alert-success">
    The print command prints the total number of characters followed by the first 100 characters of the content of all the texts read from all the 7 books for illustration purposes
</div>

In [129]:
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 6491209
FOR JESSICA, WHO LOVES STORIES,

FOR ANNE, WHO LOVED THEM TOO;

AND FOR DI, WHO HEARD THIS ONE FIRS


<div class = "alert alert-block alert-success">
    Our goal is to tokenize this 6,765,174 (~6.7m) character story into individual words and special characters that we can then turn into embeddings for training.
</div>

<div class="alert alert-warning">
    Note: It is common to process millions of articles and hundreds of thousands of books -- many gigabytes of text -- when working with LLMs. However, for educational purposes, it's sufficient to work with smaller text samples like a single book to illustrate the main ideas behind the text processing steps and to make it possible to run it in reasonable time on consumer hardware.
</div>

<div class="alert alert-block alert-danger">
    How can we best split this text to obtain a list of tokens? For this, we go on a small excursion and use Python's regular expression library re for illustration purposes. (Note that you don't have to learn or memorize any regular expression syntax since we will transition to a pre-built tokenizer later in this chapter.) 
</div>

<div class="alert alert-block alert-warning">
    Using some simple example text, we can use the re.split command with the following syntax to split a text on whitespace characters:
</div>

In [130]:
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


<div class="alert alert-block alert-info">
    The result is a list of individual words, whitespaces, and punctuation characters:
</div>

<div class="alert alert-block alert-warning">
    Let's modify the regular expression splits on whitespaces (\s) and commas, and periods ([,.]):
</div>

In [131]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


<div class="alert alert-block alert-info">
    We can see that the words and punctuation characters are now separate list entries just as we wanted
</div>

<div class="alert alert-block alert-warning">
    A small remaining issue is that the list still includes whitespace characters. Optionally, we can remove these redundant characters safely as follows:
</div>

In [132]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


<div class="alert alert-block alert-info">

REMOVING WHITESPACES OR NOT

When developing a simple tokenizer, whether we should encode whitespaces as
separate characters or just remove them depends on our application and its
requirements. Removing whitespaces reduces the memory and computing
requirements. However, keeping whitespaces can be useful if we train models that
are sensitive to the exact structure of the text (for example, Python code, which is
sensitive to indentation and spacing). Here, we remove whitespaces for simplicity
and brevity of the tokenized outputs. Later, we will switch to a tokenization scheme
that includes whitespaces.

</div>

<div class="alert alert-block alert-warning">
    The tokenization scheme we devised above works well on the simple sample text. Let's modify it a bit further so that it can also handle other types of punctuation, such as question marks, quotation marks, and the double-dashes we have seen earlier in the first 100 characters of Harry Potter story, along with additional special characters: 
</div>

In [133]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [134]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [135]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([-,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '-', '-', 'a', 'test', '?']


<div class="alert alert-block alert-success">
    Now that we got a basic tokenizer working, let's apply it to our Harry Potter books:
</div>

In [136]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['FOR', 'JESSICA', ',', 'WHO', 'LOVES', 'STORIES', ',', 'FOR', 'ANNE', ',', 'WHO', 'LOVED', 'THEM', 'TOO', ';', 'AND', 'FOR', 'DI', ',', 'WHO', 'HEARD', 'THIS', 'ONE', 'FIRST', '.', 'CONTENTS', 'ONE', 'The', 'Boy', 'Who']


In [137]:
print(len(preprocessed))

1366092


### Step 2: Creating Token IDs

<div class="alert alert-block alert-warning">
    In the previous section, we tokenized all Harry Potter books and assigned it to a Python variable called preprocessed. Let's now create a list of all unique tokens and sort them alphabetically to determine the vocabulary size:
</div>

In [138]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

31066


<div class="alert alert-block alert-success">
    After determining that the vocabulary size is 31066 via the above code, we create the vocabulary and print its first 50 entries for illustration purposes:
</div>

In [139]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [149]:
for i, item in enumerate(vocab.items()):
    if i > 100 and i < 119:
        print(item)
    if i >= 120:
        break

('ABOUT', 101)
('ABSOLUTELY', 102)
('ACCEPTABLE', 103)
('ACCIDENTS', 104)
('ACCIO', 105)
('ACT', 106)
('ADVANCE', 107)
('ADVERTISE', 108)
('ADVICE', 109)
('AFTER', 110)
('AGAIN', 111)
('AGAINST', 112)
('AGUAMENTI', 113)
('AID', 114)
('AINE', 115)
('ALBUS', 116)
('ALBUS’s', 117)
('ALIVE', 118)


<div class="alert alert-block alert-info">
    As we can see, based on the output above, the dictionary contains individual tokens associated with unique integer labels. 
</div>

<div class="alert alert-block alert-success">

Later in this book, when we want to convert the outputs of an LLM from numbers back into text, we also need a way to turn token IDs into text. 

For this, we can create an inverse version of the vocabulary that maps token IDs back to corresponding text tokens.

</div>

<div class="alert alert-block alert-success">

Let's implement a complete tokenizer class in Python.

The class will have an encode method that splits
text into tokens and carries out the string-to-integer mapping to produce token IDs via the
vocabulary. 

In addition, we implement a decode method that carries out the reverse
integer-to-string mapping to convert the token IDs back into text.

</div>

<div class="alert alert-block alert-info">
    
Step 1: Store the vocabulary as a class attribute for access in the encode and decode methods
    
Step 2: Create an inverse vocabulary that maps token IDs back to the original text tokens

Step 3: Process input text into token IDs

Step 4: Convert token IDs back into text

Step 5: Replace spaces before the specified punctuation

</div>

In [150]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

<div class="alert alert-block alert-success">
    Let's instantiate a new tokenizer object from the SimpleTokenizerV1 class and tokenize a passage from Harry Potter's story to try it out in practice:
</div>

In [151]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
        The Dursleys had everything they wanted, but they also had a secret, and their greatest fear was that somebody would discover it
       """
ids = tokenizer.encode(text)
print(ids)

[5784, 1721, 14654, 12587, 24967, 26739, 5, 8940, 24967, 7040, 14654, 6607, 22184, 5, 7105, 24935, 14445, 13014, 26783, 24918, 23288, 27398, 11444, 16229]


<div class="alert alert-block alert-info">
    The code above prints the following token IDs: <br>
    Next, let's see if we can turn these token IDs back into text using the decode method:
</div>

In [152]:
tokenizer.decode(ids)

'The Dursleys had everything they wanted, but they also had a secret, and their greatest fear was that somebody would discover it'

<div class="alert alert-block alert-info">
    Based on the output above, we can see that the decode method successfully converted the token IDs back into the original text.
</div>

<div class="alert alert-block alert-success">

So far, so good. We implemented a tokenizer capable of tokenizing and de-tokenizing text based on a snippet from the training set. 

Let's now apply it to a new text sample that is not contained in the training set:

</div>

In [155]:
text = "Hello, do you like chai?"
print(tokenizer.encode(text))

KeyError: 'chai'

<div class="alert alert-block alert-danger">
    
The problem is that the word "chai" was not used in the Harry Potter books. 

Hence, it is not contained in the vocabulary. 

This highlights the need to consider large and diverse training sets to extend the vocabulary when working on LLMs.

</div>

### ADDING SPECIAL CONTEXT TOKENS

In the previous section, we implemented a simple tokenizer and applied it to a passage
from the training set. 

In this section, we will modify this tokenizer to handle unknown
words.


In particular, we will modify the vocabulary and tokenizer we implemented in the
previous section, SimpleTokenizerV2, to support two new tokens, <|unk|> and
<|endoftext|>

<div class="alert alert-block alert-warning">

We can modify the tokenizer to use an <|unk|> token if it encounters a word that is not part of the vocabulary. 

Furthermore, we add a token between unrelated texts. 

For example, when training GPT-like LLMs on multiple independent documents or books, it is common to insert a token before each document or book that follows a previous text source

</div>

<div class="alert alert-block alert-success">
    Let's now modify the vocabulary to include these two special tokens, <unk> and <|endoftext|>, by adding these to the list of all unique words that we created in the previous section:
</div>

In [156]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [157]:
len(vocab.items())

31068

<div class="alert alert-block alert-info">
    
Based on the output of the print statement above, the new vocabulary size is 31068 (the vocabulary size in the previous section was 31066).

</div>

<div class="alert alert-block alert-success">
    As an additional quick check, let's print the last 5 entries of the updated vocabulary:
</div>

In [158]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('”', 31063)
('”s', 31064)
('”said', 31065)
('<|endoftext|>', 31066)
('<|unk|>', 31067)


<div class="alert alert-block alert-success">
    A simple text tokenizer that handles unknown words
</div>

<div class="alert alert-block alert-info">

Step 1: Replace unknown words by <|unk|> tokens

Step 2: Replace spaces before the specified punctuations

</div>

In [159]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [160]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like chai?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like chai? <|endoftext|> In the sunlit terraces of the palace.


In [161]:
tokenizer.encode(text)

[2761,
 5,
 11658,
 27537,
 16928,
 31067,
 93,
 31066,
 3006,
 24928,
 24332,
 31067,
 18622,
 24928,
 19076,
 8]

In [162]:
tokenizer.decode(tokenizer.encode(text))

'Hello, do you like <|unk|>? <|endoftext|> In the sunlit <|unk|> of the palace.'

<div class="alert alert-block alert-info">
    Based on comparing the de-tokenized text above with the original input text, we know that the training dataset, Harry Potter story, did not contain the words "chai" and "terraces."
</div>

<div class="alert alert-block alert-warning">

So far, we have discussed tokenization as an essential step in processing text as input to
LLMs. Depending on the LLM, some researchers also consider additional special tokens such
as the following:

[BOS] (beginning of sequence): This token marks the start of a text. It
signifies to the LLM where a piece of content begins.

[EOS] (end of sequence): This token is positioned at the end of a text,
and is especially useful when concatenating multiple unrelated texts,
similar to <|endoftext|>. For instance, when combining two different
Wikipedia articles or books, the [EOS] token indicates where one article
ends and the next one begins.

[PAD] (padding): When training LLMs with batch sizes larger than one,
the batch might contain texts of varying lengths. To ensure all texts have
the same length, the shorter texts are extended or "padded" using the
[PAD] token, up to the length of the longest text in the batch.

</div>


<div class="alert alert-block alert-warning">
    Note that the tokenizer used for GPT models does not need any of these tokens mentioned above but only uses an <|endoftext|> token for simplicity
</div>

<div class="alert alert-block alert-warning">
    The tokenizer used for GPT models also doesn't use an <|unk|> token for outof-vocabulary words. Instead, GPT models use a byte pair encoding tokenizer, which breaks down words into subword units
</div>